In [1]:
from datasets import load_dataset
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_test = load_dataset('glue', 'mrpc', split='test')
dataset_val = load_dataset('glue', 'mrpc', split='validation')

Reusing dataset glue (C:\Users\sago\.cache\huggingface\datasets\glue\mrpc\1.0.0\dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
Reusing dataset glue (C:\Users\sago\.cache\huggingface\datasets\glue\mrpc\1.0.0\dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
Reusing dataset glue (C:\Users\sago\.cache\huggingface\datasets\glue\mrpc\1.0.0\dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [3]:
def encode(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length')

dataset_train = dataset_train.map(encode, batched=True)
#Rename label to labels
dataset_train = dataset_train.map(lambda rows: {'labels': rows['label']}, batched=True)

Parameter 'function'=<function encode at 0x0000024B46DB7F70> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


  0%|          | 0/4 [00:00<?, ?ba/s]

  0%|          | 0/4 [00:00<?, ?ba/s]

In [4]:
import torch
dataset_train.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
dataloader = torch.utils.data.DataLoader(dataset_train, batch_size=64)

In [5]:
#Train model
from tqdm import tqdm
#device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = 'cpu'
print(f'device is {device}')
model.train().to(device)
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-5)
for epoch in range(2):
    for i, batch in enumerate(tqdm(dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs[0]
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if i % 10 == 0:
            print(f"loss: {loss}")

device is cpu


  2%|▏         | 1/58 [03:46<3:35:09, 226.49s/it]

loss: 0.8163777589797974


 19%|█▉        | 11/58 [40:06<2:50:33, 217.74s/it]

loss: 0.6952430605888367


...

 71%|███████   | 41/58 [2:29:09<1:01:55, 218.58s/it]

loss: 0.50560063123703


 88%|████████▊ | 51/58 [3:05:50<25:37, 219.69s/it]  

loss: 0.5051382184028625


100%|██████████| 58/58 [3:29:26<00:00, 216.67s/it]


In [6]:
torch.save(model.state_dict(), 'state.pt')
torch.save(model, 'model.pt')